In [1]:
%%writefile tools/verification.py
"""Targeted corroboration (Day 25). For claims backed by fewer than the required number of
independent domains, search for the claim itself, excluding the domain(s) already cited,
and accept a new source ONLY if one of its sentences actually states the claim.
No LLM calls: this spends search requests, not Groq output tokens."""
import os
import re
from typing import Callable, Dict, List, Optional, Tuple

from tools.claim_merge import _profile
from tools.evidence import (independent_domains, independent_source_count,
                            MIN_INDEPENDENT_SOURCES)

MAX_VERIFY_CLAIMS = 6            # per run, keeps search volume inside free-tier limits
OVERLAP_WITH_NUMBERS = 0.5       # claim has figures -> the same figures must appear in the sentence
OVERLAP_NO_NUMBERS = 0.75        # no figures to anchor on -> demand much closer wording


def enabled() -> bool:
    return os.getenv("VERIFY_BY_SEARCH", "false").lower() == "true"


def _sentences(text: str) -> List[str]:
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text or "") if s.strip()]


def matching_sentence(claim: str, text: str) -> Optional[str]:
    """The first sentence of `text` that states the claim (same figures, same polarity,
    enough shared content words), or None. Sentence-level so a stray 'not' elsewhere can't flip it."""
    p = _profile(claim)
    if not p["tokens"]:
        return None
    threshold = OVERLAP_WITH_NUMBERS if p["numbers"] else OVERLAP_NO_NUMBERS
    for sentence in _sentences(text):
        s = _profile(sentence)
        if p["numbers"] and not p["numbers"] <= s["numbers"]:
            continue
        if s["negated"] != p["negated"]:
            continue
        if len(p["tokens"] & s["tokens"]) / len(p["tokens"]) >= threshold:
            return sentence
    return None


def supports(claim: str, text: str) -> bool:
    return matching_sentence(claim, text) is not None


def verify_claim(fact: Dict, search_fn: Callable) -> Tuple[Dict, int, bool]:
    """Returns (fact, sources_added, search_failed)."""
    have = independent_domains(fact.get("sources", []))
    try:
        candidates = search_fn(fact["claim"], exclude_domains=sorted(have))
    except Exception:
        return fact, 0, True

    sources = list(fact.get("sources", []))
    added = 0
    for c in candidates or []:
        new_domain = independent_domains([c]) - have
        sentence = matching_sentence(fact["claim"], c.get("content", "")) if new_domain else None
        if sentence:
            have |= new_domain
            sources.append({"id": len(sources) + 1, "title": c.get("title", ""),
                            "url": c.get("url", ""), "via": "verification",
                            "evidence": sentence.strip()[:300]})
            added += 1
    if not added:
        return fact, 0, False
    return {**fact, "sources": sources, "verified_by_search": added}, added, False


def _priority_order(facts: List[Dict], needy: List[int]) -> List[int]:
    """Round-robin across sub-questions so one topic can't use up the whole verification budget;
    inside each topic, claims with figures go first."""
    groups: Dict = {}
    for i in needy:
        groups.setdefault(facts[i].get("sub_question"), []).append(i)
    queues = []
    for idxs in groups.values():
        idxs.sort(key=lambda i: (not _profile(facts[i]["claim"])["numbers"], i))
        queues.append(idxs)
    order = []
    while any(queues):
        for q in queues:
            if q:
                order.append(q.pop(0))
    return order


def verify_claims(facts: List[Dict], search_fn: Callable,
                  max_claims: int = MAX_VERIFY_CLAIMS) -> Tuple[List[Dict], Dict]:
    """Verify the most checkable under-sourced claims (those with figures first), sequentially."""
    needy = [i for i, f in enumerate(facts)
             if independent_source_count(f.get("sources", [])) < MIN_INDEPENDENT_SOURCES]
    needy = _priority_order(facts, needy)

    out = list(facts)
    stats = {"tried": 0, "upgraded": 0, "sources_added": 0, "failed": 0}
    for i in needy[:max_claims]:
        stats["tried"] += 1
        new_fact, added, failed = verify_claim(out[i], search_fn)
        stats["failed"] += int(failed)
        stats["sources_added"] += added
        if independent_source_count(new_fact.get("sources", [])) >= MIN_INDEPENDENT_SOURCES:
            stats["upgraded"] += 1
        out[i] = new_fact
    return out, stats

Overwriting tools/verification.py


In [2]:
%%writefile tests/test_verification.py
from tools.verification import supports, verify_claim, verify_claims

CLAIM = "Hydrometallurgical processing achieves a 95% recovery rate for lithium."


def cand(url, text, title="t"):
    return {"title": title, "url": url, "content": text}


def test_supports_requires_same_figures():
    assert supports(CLAIM, "Hydrometallurgical processes reach a 95% recovery rate for lithium in tests.")
    assert not supports(CLAIM, "Hydrometallurgical processes reach a 70% recovery rate for lithium.")


def test_supports_is_sentence_level_for_negation():
    text = "This is not a small market. Hydrometallurgical processing achieves a 95% recovery rate for lithium."
    assert supports(CLAIM, text)           # the stray 'not' is in a different sentence
    assert not supports(CLAIM, "Hydrometallurgical processing does not achieve a 95% recovery rate for lithium.")


def test_claims_without_figures_need_close_wording():
    claim = "Redwood Materials operates the largest lithium-ion battery recycling facility outside Asia"
    assert not supports(claim, "Redwood Materials launched a second-life battery storage partnership.")
    assert supports(claim, "Redwood Materials operates the largest lithium-ion battery recycling facility outside Asia today.")


def test_verify_adds_only_new_supporting_domains():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}
    calls = []

    def search(q, exclude_domains=()):
        calls.append(exclude_domains)
        return [cand("https://a.org/y", "Hydrometallurgical processing achieves a 95% recovery rate for lithium."),
                cand("https://b.com/z", "Hydrometallurgical processing achieves a 95% recovery rate for lithium."),
                cand("https://c.net/q", "Unrelated text about cathodes.")]

    new, added, failed = verify_claim(fact, search)
    assert calls == [["a.org"]] and added == 1 and not failed
    assert [s["url"] for s in new["sources"]] == ["https://a.org/x", "https://b.com/z"]
    assert new["sources"][1]["via"] == "verification" and [s["id"] for s in new["sources"]] == [1, 2]


def test_search_failure_leaves_fact_unchanged():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}

    def boom(q, exclude_domains=()):
        raise RuntimeError("down")

    new, added, failed = verify_claim(fact, boom)
    assert new is fact and added == 0 and failed


def test_verify_claims_prioritises_figures_and_respects_cap():
    def mk(claim, n):
        return {"claim": claim, "sources": [{"id": 1, "title": "a", "url": f"https://s{n}.org/x"}]}
    facts = [mk("Plain statement about recycling policy and regulation across regions here", 1),
             mk("Plant capacity reaches 50000 tonnes per year in Nevada operations", 2),
             mk("Another plain statement about supply chain integration strategy overall", 3)]
    seen = []

    def search(q, exclude_domains=()):
        seen.append(q)
        return []

    _, stats = verify_claims(facts, search, max_claims=2)
    assert stats["tried"] == 2 and seen[0].startswith("Plant capacity")


def test_already_corroborated_claims_not_searched():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"},
                                         {"id": 2, "title": "b", "url": "https://b.com/x"}]}
    _, stats = verify_claims([fact], lambda q, exclude_domains=(): 1 / 0)
    assert stats["tried"] == 0


def test_upgraded_counted_when_claim_reaches_two_domains():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}
    search = lambda q, exclude_domains=(): [cand("https://b.com/z", CLAIM)]
    out, stats = verify_claims([fact], search)
    assert stats == {"tried": 1, "upgraded": 1, "sources_added": 1, "failed": 0}


def test_verification_records_the_evidence_sentence():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}
    text = "Intro text. Hydrometallurgical processing achieves a 95% recovery rate for lithium. Outro."
    new, _, _ = verify_claim(fact, lambda q, exclude_domains=(): [cand("https://b.com/z", text)])
    assert new["sources"][1]["evidence"] == "Hydrometallurgical processing achieves a 95% recovery rate for lithium."


def test_verification_budget_is_shared_across_subquestions():
    def mk(sq, n):
        return {"claim": f"Plant {sq} capacity reaches {n}0000 tonnes per year at site number {n}",
                "sub_question": sq,
                "sources": [{"id": 1, "title": "a", "url": f"https://{sq.lower()}{n}.org/x"}]}
    facts = [mk("Q1", n) for n in range(1, 5)] + [mk("Q2", n) for n in range(1, 5)]
    tried = []

    def search(q, exclude_domains=()):
        tried.append(q)
        return []

    verify_claims(facts, search, max_claims=4)
    assert sum("Q1" in q for q in tried) == 2 and sum("Q2" in q for q in tried) == 2

Overwriting tests/test_verification.py


In [3]:
%%writefile tools/metrics.py
"""Run-level observability: tokens, latency and errors per node, plus evidence stats."""
import json
import threading
import time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler

RATE_LIMIT_MARKERS = ("429", "rate limit", "rate_limit")


def peak_tokens_in_window(events, window=60.0):
    """events: [(timestamp, output_tokens)]. Max token sum inside any `window`-second span."""
    events = sorted(events)
    best = cur = lo = 0
    for hi in range(len(events)):
        cur += events[hi][1]
        while events[hi][0] - events[lo][0] > window:
            cur -= events[lo][1]
            lo += 1
        best = max(best, cur)
    return best


class RunMetrics(BaseCallbackHandler):
    """Pass as config={"callbacks": [metrics]} to graph.invoke."""

    def __init__(self):
        self._lock = threading.Lock()
        self._llm_runs = {}    # run_id -> (node, start)
        self._node_runs = {}   # run_id -> (node, start)
        self.llm_calls = []
        self.node_times = defaultdict(list)
        self.node_errors = Counter()
        self.output_events = []   # (timestamp, output_tokens)
        self.error_log = []       # (node, kind, message) for every failed LLM call
        self.started = time.time()

    @staticmethod
    def _node(metadata):
        return (metadata or {}).get("langgraph_node", "unknown")

    # ---- LLM calls ----
    def on_chat_model_start(self, serialized, messages, *, run_id, metadata=None, **kwargs):
        with self._lock:
            self._llm_runs[run_id] = (self._node(metadata), time.time())

    def on_llm_start(self, serialized, prompts, *, run_id, metadata=None, **kwargs):
        with self._lock:
            self._llm_runs[run_id] = (self._node(metadata), time.time())

    def on_llm_end(self, response, *, run_id, **kwargs):
        now = time.time()
        with self._lock:
            node, start = self._llm_runs.pop(run_id, ("unknown", now))
        inp = out = reasoning = 0
        try:
            msg = response.generations[0][0].message
            um = getattr(msg, "usage_metadata", None) or {}
            inp = um.get("input_tokens", 0)
            out = um.get("output_tokens", 0)
            reasoning = (um.get("output_token_details") or {}).get("reasoning", 0)
        except (IndexError, AttributeError):
            pass
        if not out:
            tu = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
            inp = inp or tu.get("prompt_tokens", 0)
            out = tu.get("completion_tokens", 0)
        with self._lock:
            self.llm_calls.append({"node": node, "input": inp, "output": out,
                                   "reasoning": reasoning, "seconds": now - start,
                                   "error": None})
            self.output_events.append((now, out))

    def on_llm_error(self, error, *, run_id, **kwargs):
        now = time.time()
        with self._lock:
            node, start = self._llm_runs.pop(run_id, ("unknown", now))
        msg = str(error).lower()
        kind = "rate_limit" if any(m in msg for m in RATE_LIMIT_MARKERS) else "other"
        self.error_log.append((node, kind, str(error)[:300]))
        with self._lock:
            self.llm_calls.append({"node": node, "input": 0, "output": 0,
                                   "reasoning": 0, "seconds": now - start,
                                   "error": kind})

    # ---- node latency (a node is a chain whose name equals its langgraph_node) ----
    def on_chain_start(self, serialized, inputs, *, run_id, metadata=None, **kwargs):
        node = (metadata or {}).get("langgraph_node")
        if node and kwargs.get("name") == node:
            with self._lock:
                self._node_runs[run_id] = (node, time.time())

    def _finish_node(self, run_id, failed=False):
        with self._lock:
            entry = self._node_runs.pop(run_id, None)
            if entry:
                node, start = entry
                self.node_times[node].append(time.time() - start)
                if failed:
                    self.node_errors[node] += 1

    def on_chain_end(self, outputs, *, run_id, **kwargs):
        self._finish_node(run_id)

    def on_chain_error(self, error, *, run_id, **kwargs):
        self._finish_node(run_id, failed=True)

    # ---- summary ----
    def summary(self):
        nodes = defaultdict(lambda: {"calls": 0, "input": 0, "output": 0, "reasoning": 0,
                                     "rate_limited": 0, "other_errors": 0,
                                     "llm_seconds": 0.0, "max_call_output": 0})
        for c in self.llm_calls:
            n = nodes[c["node"]]
            n["calls"] += 1
            n["input"] += c["input"]
            n["output"] += c["output"]
            n["reasoning"] += c["reasoning"]
            n["llm_seconds"] += c["seconds"]
            n["max_call_output"] = max(n["max_call_output"], c["output"])
            if c["error"] == "rate_limit":
                n["rate_limited"] += 1
            elif c["error"]:
                n["other_errors"] += 1
        for node, times in self.node_times.items():
            nodes[node]["node_runs"] = len(times)
            nodes[node]["node_seconds"] = sum(times)
            nodes[node]["slowest_run"] = max(times)
            nodes[node]["node_errors"] = self.node_errors[node]
        return {
            "total_seconds": time.time() - self.started,
            "total_output_tokens": sum(c["output"] for c in self.llm_calls),
            "total_reasoning_tokens": sum(c["reasoning"] for c in self.llm_calls),
            "peak_output_tokens_60s": peak_tokens_in_window(self.output_events),
            "rate_limit_errors": sum(1 for c in self.llm_calls if c["error"] == "rate_limit"),
            "nodes": {k: dict(v) for k, v in nodes.items()},
        }


def evidence_stats(state):
    """Evidence picture from the final state: claims before/after merging, cited vs
    independent sources per claim, and search-provider mix."""
    cited, independent, status = Counter(), Counter(), Counter()
    results = state.get("fact_check_results", []) or []
    for f in results:
        cited[len(f.get("sources") or f.get("source_ids") or [])] += 1
        independent[f.get("independent_sources", "n/a")] += 1
        status[f.get("status") or f.get("verdict") or "n/a"] += 1
    extracted = state.get("extracted_facts", []) or []
    merged_groups = [f for f in results if f.get("merged_count", 1) > 1]
    providers = Counter(r.get("provider", "n/a") for r in (state.get("search_results", []) or [])
                        if isinstance(r, dict))
    return {"extracted_claims": len(extracted),
            "claims_after_merge_expand": len(results),
            "claims_that_were_merged": len(merged_groups),
            "cited_per_claim": dict(sorted(cited.items())),
            "independent_per_claim": dict(independent),
            "status": dict(status),
            "search_providers": dict(providers),
            "claims_verified_by_search": sum(1 for f in results if f.get("verified_by_search"))}


def save_run(metrics, state, label="run"):
    Path("data/runs").mkdir(parents=True, exist_ok=True)
    path = Path("data/runs") / f"{datetime.now():%Y%m%d_%H%M%S}_{label}.json"
    path.write_text(json.dumps({"metrics": metrics.summary(),
                                "evidence": evidence_stats(state)}, indent=2, default=str))
    return path


def print_run_report(metrics, state):
    s = metrics.summary()
    print(f"Total: {s['total_seconds']:.0f}s | output tokens: {s['total_output_tokens']} "
          f"(reasoning: {s['total_reasoning_tokens']}) | 429s: {s['rate_limit_errors']}")
    print(f"Peak output tokens in any 60s window: {s['peak_output_tokens_60s']} (Groq limit: 1000)\n")
    print(f"{'node':<22}{'calls':>6}{'out':>7}{'reason':>8}{'max/call':>9}{'429':>5}{'node_s':>8}")
    for name, n in s["nodes"].items():
        print(f"{name:<22}{n['calls']:>6}{n['output']:>7}{n['reasoning']:>8}"
              f"{n['max_call_output']:>9}{n['rate_limited']:>5}{n.get('node_seconds', 0):>8.1f}")
    ev = evidence_stats(state)
    print("\nEvidence")
    print(f"  claims extracted: {ev['extracted_claims']} -> after merge/expand: "
          f"{ev['claims_after_merge_expand']} ({ev['claims_that_were_merged']} rows from merged groups)")
    print("  cited sources per claim:     ", ev["cited_per_claim"])
    print("  independent domains per claim:", ev["independent_per_claim"])
    print("  status:                       ", ev["status"])
    print("  search providers:             ", ev["search_providers"])
    print("  claims given extra sources by verification search:", ev["claims_verified_by_search"])
    for node, kind, msg in metrics.error_log:
        print(f"  LLM error in {node} ({kind}): {msg}")

Overwriting tools/metrics.py


In [4]:
import os
os.makedirs("ui", exist_ok=True)
open("ui/__init__.py", "a").close()

In [5]:
%%writefile ui/progress.py
"""Turns LangGraph 'updates' chunks into human-readable progress lines (Day 26).
Pure Python (no Streamlit import) so it can be unit-tested."""
from collections import Counter
from typing import Dict, List, Optional, Tuple

STAGES = ["web_search_single", "data_extraction", "fact_checking", "summarization", "report_generator"]
STAGE_LABELS = {
    "supervisor": "Supervisor",
    "web_search_single": "Web search",
    "data_extraction": "Data extraction",
    "fact_checking": "Fact-checking",
    "summarization": "Summarization",
    "report_generator": "Report generator",
}


def stage_progress(node: str) -> Optional[float]:
    """0..1 progress once `node` has reported (None for nodes outside the pipeline)."""
    if node not in STAGES:
        return None
    return (STAGES.index(node) + 1) / len(STAGES)


def describe_update(node: str, update) -> str:
    """One short line describing what a node just did."""
    update = update or {}
    errors = update.get("errors") or []
    suffix = f"  ⚠ {len(errors)} error(s)" if errors else ""

    if node == "supervisor":
        plan = update.get("plan")
        if plan:
            return f"Planned {len(plan)} sub-questions" + suffix
        nxt = update.get("next_agent")
        return (f"Next step: {STAGE_LABELS.get(nxt, nxt)}" if nxt else "Supervisor decided the next step") + suffix

    if node == "web_search_single":
        lines = []
        for r in update.get("search_results") or []:
            q = (r.get("sub_question") or "")[:70]
            if r.get("status") == "failed" or not r.get("sources"):
                lines.append(f"Search found nothing for “{q}”")
            else:
                lines.append(f"Searched “{q}” via {r.get('provider', '?')} ({len(r['sources'])} sources)")
        return "; ".join(lines) + suffix if lines else "Search finished" + suffix

    if node == "data_extraction":
        return f"Extracted {len(update.get('extracted_facts') or [])} claims" + suffix

    if node == "fact_checking":
        results = update.get("fact_check_results") or []
        c = Counter(r.get("status", "UNVERIFIED") for r in results)
        return (f"Checked {len(results)} claims: {c.get('CONFIRMED', 0)} confirmed, "
                f"{c.get('DISPUTED', 0)} disputed, {c.get('UNVERIFIED', 0)} unverified") + suffix

    if node == "summarization":
        return f"Summarised {len(update.get('summaries') or [])} sub-topic(s)" + suffix

    if node == "report_generator":
        report = update.get("final_report") or ""
        return f"Report written ({len(report)} characters)" + suffix

    return f"{node} finished" + suffix


def iter_updates(chunk) -> List[Tuple[str, Dict]]:
    """Normalises one 'updates' chunk -> [(node, update)], tolerating None / list updates."""
    out = []
    if not isinstance(chunk, dict):
        return out
    for node, update in chunk.items():
        if isinstance(update, list):
            out.extend((node, u) for u in update)
        else:
            out.append((node, update))
    return out

Writing ui/progress.py


In [6]:
%%writefile tests/test_progress.py
from ui.progress import describe_update, stage_progress, iter_updates


def test_supervisor_plan_and_routing():
    assert describe_update("supervisor", {"plan": ["a", "b", "c"]}) == "Planned 3 sub-questions"
    assert describe_update("supervisor", {"next_agent": "fact_checking"}) == "Next step: Fact-checking"
    assert describe_update("supervisor", None) == "Supervisor decided the next step"


def test_search_line_reports_provider_and_failures():
    ok = {"search_results": [{"sub_question": "How does X work?", "provider": "tavily", "sources": [1, 2, 3, 4]}]}
    bad = {"search_results": [{"sub_question": "Q", "provider": "none", "sources": [], "status": "failed"}]}
    assert "via tavily (4 sources)" in describe_update("web_search_single", ok)
    assert "found nothing" in describe_update("web_search_single", bad)


def test_fact_check_counts_statuses():
    u = {"fact_check_results": [{"status": "CONFIRMED"}, {"status": "UNVERIFIED"}, {"status": "UNVERIFIED"}]}
    assert describe_update("fact_checking", u) == "Checked 3 claims: 1 confirmed, 0 disputed, 2 unverified"


def test_errors_are_flagged_and_unknown_nodes_fall_back():
    assert "⚠ 1 error(s)" in describe_update("data_extraction", {"extracted_facts": [], "errors": [{"x": 1}]})
    assert describe_update("mystery", {}) == "mystery finished"


def test_progress_fraction_only_for_pipeline_nodes():
    assert stage_progress("supervisor") is None
    assert stage_progress("web_search_single") == 0.2 and stage_progress("report_generator") == 1.0


def test_iter_updates_tolerates_odd_shapes():
    assert iter_updates({"a": {"x": 1}, "b": None}) == [("a", {"x": 1}), ("b", None)]
    assert iter_updates({"a": [{"x": 1}, {"x": 2}]}) == [("a", {"x": 1}), ("a", {"x": 2})]
    assert iter_updates("not a dict") == []

Writing tests/test_progress.py


In [7]:
%%writefile app.py
"""Streamlit front end: live agent progress, report, evidence table, run metrics (Day 26).
Run from a terminal in the project root:  streamlit run app.py"""
import os
import time

import streamlit as st

from tools.metrics import RunMetrics, evidence_stats, save_run
from ui.progress import STAGE_LABELS, describe_update, iter_updates, stage_progress

st.set_page_config(page_title="Multi-Agent Research System", layout="wide")
st.title("Multi-Agent Research & Intelligence System")

with st.sidebar:
    st.header("Options")
    verify = st.checkbox("Verify single-source claims by search", value=True,
                         help="Searches for a second independent source for up to 6 claims per run, "
                              "spread across sub-topics.")
    use_cache = st.checkbox("Cache search results (24h)", value=True,
                            help="Repeated queries skip the search APIs.")
    st.caption("Set DEMO_MODE=true in .env to run searches one at a time.")
os.environ["VERIFY_BY_SEARCH"] = "true" if verify else "false"
os.environ["SEARCH_CACHE"] = "true" if use_cache else "false"


@st.cache_resource
def get_graph():
    from graph.build_graph import build_graph
    return build_graph()


query = st.text_input("Research question",
                      value="Compare battery recycling approaches of Tesla, Redwood Materials and CATL")

if st.button("Run research", type="primary", disabled=not query.strip()):
    try:
        graph = get_graph()
    except Exception as e:
        st.error(f"Could not start the graph: {type(e).__name__}: {e}")
        st.stop()

    metrics = RunMetrics()
    initial_state = {
        "query": query.strip(), "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    bar = st.progress(0.0)
    started, final = time.time(), None
    try:
        with st.status("Running agents…", expanded=True) as status:
            for mode, chunk in graph.stream(initial_state, config={"callbacks": [metrics]},
                                            stream_mode=["updates", "values"]):
                if mode == "values":
                    final = chunk
                    continue
                for node, update in iter_updates(chunk):
                    st.write(f"**{STAGE_LABELS.get(node, node)}** — {describe_update(node, update)}")
                    frac = stage_progress(node)
                    if frac:
                        bar.progress(frac)
            status.update(label=f"Finished in {time.time() - started:.0f}s", state="complete")
    except Exception as e:
        st.error(f"Run failed: {type(e).__name__}: {e}")
        st.stop()

    if final:
        save_run(metrics, final, label="ui")
        st.session_state["last"] = {"final": final, "metrics": metrics.summary(),
                                    "evidence": evidence_stats(final), "errors": metrics.error_log}

last = st.session_state.get("last")
if last:
    final = last["final"]
    tab_report, tab_evidence, tab_metrics = st.tabs(["Report", "Evidence", "Run metrics"])

    with tab_report:
        report = final.get("final_report") or "_No report was produced._"
        st.markdown(report)
        st.download_button("Download report (.md)", report, file_name="report.md")

    with tab_evidence:
        ev = last["evidence"]
        c1, c2, c3 = st.columns(3)
        c1.metric("Claims", ev["claims_after_merge_expand"])
        c2.metric("Confirmed", ev["status"].get("CONFIRMED", 0))
        c3.metric("Upgraded by verification search", ev["claims_verified_by_search"])
        results = final.get("fact_check_results", [])
        st.dataframe([{"claim": f.get("claim"), "status": f.get("status"),
                       "independent domains": f.get("independent_sources"),
                       "verified by search": bool(f.get("verified_by_search")),
                       "sources": ", ".join(s.get("url", "") for s in f.get("sources", []))}
                      for f in results], use_container_width=True)

        verified = {f["claim"]: f for f in results if f.get("verified_by_search")}
        if verified:
            with st.expander(f"How verification found its extra sources ({len(verified)} claims)"):
                for claim, f in verified.items():
                    st.markdown(f"**{claim}** — {f.get('status')}")
                    for s in f.get("sources", []):
                        if s.get("via") == "verification":
                            st.markdown(f"- [{s.get('url')}]({s.get('url')})  \n  “{s.get('evidence', '')}”")

    with tab_metrics:
        m = last["metrics"]
        c1, c2, c3 = st.columns(3)
        c1.metric("Total time", f"{m['total_seconds']:.0f}s")
        c2.metric("Output tokens", m["total_output_tokens"])
        c3.metric("Rate-limit errors", m["rate_limit_errors"])
        st.caption(f"Peak output tokens in any 60s window: {m['peak_output_tokens_60s']} (Groq free tier limit: 1000)")
        st.dataframe([{"node": k, **v} for k, v in m["nodes"].items()], use_container_width=True)
        for node, kind, msg in last["errors"]:
            st.warning(f"LLM error in {node} ({kind}): {msg}")

Writing app.py


In [8]:
import os
os.environ["VERIFY_BY_SEARCH"] = "true"
os.environ["SEARCH_CACHE"] = "true"

from graph.build_graph import build_graph
from tools.metrics import RunMetrics, print_run_report, save_run
from tools import cache

app = build_graph()
query = "Compare battery recycling approaches of Tesla, Redwood Materials and CATL"
initial_state = {
    "query": query, "plan": [], "next_agent": "", "iteration_count": 0,
    "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
    "model_usage_log": [],
}

metrics = RunMetrics()
result = app.invoke(initial_state, config={"callbacks": [metrics]})
print_run_report(metrics, result)
print("cache:", cache.stats())
print("saved:", save_run(metrics, result, label="day26"))

print("\n\n=== AUDIT: claims upgraded by verification search ===")
seen = set()
for f in result["fact_check_results"]:
    if f.get("verified_by_search") and f["claim"] not in seen:
        seen.add(f["claim"])
        print(f"\nCLAIM: {f['claim']}  ->  {f['status']}")
        for s in f["sources"]:
            tag = "new " if s.get("via") == "verification" else "orig"
            print(f"  [{tag}] {s['url']}")
            if s.get("evidence"):
                print(f"         \"{s['evidence']}\"")

2026-11-03 18:04:22,380 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
2026-11-03 18:04:22,384 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 18:04:22,387 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 18:04:22,389 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 18:04:22,399 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 18:04:22,402 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-11-03 18:04:24,027 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the primary technical processes and recovery rates for lithium, cobalt, and nickel in the recycling facilities operated by Tesla, Redwood Materials, and CATL? 

Total: 183s | output tokens: 4833 (reasoning: 0) | 429s: 0
Peak output tokens in any 60s window: 2064 (Groq limit: 1000)

node                   calls    out  reason max/call  429  node_s
supervisor                 1    127       0      127    0     0.7
data_extraction            4   1937       0      604    0    58.1
fact_checking              3   1278       0      491    0    73.5
summarization              4    639       0      176    0    47.4
report_generator           1    852       0      852    0     3.2
web_search_single          0      0       0        0    0     0.0

Evidence
  claims extracted: 43 -> after merge/expand: 38 (3 rows from merged groups)
  cited sources per claim:      {1: 31, 2: 3, 3: 1, 5: 3}
  independent domains per claim: {1: 31, 3: 1, 2: 3, 5: 3}
  status:                        {'UNVERIFIED': 31, 'CONFIRMED': 7}
  search providers:              {'tavily': 4}
  claims given extra sources by verification search: 5
cache: {'hits': 8, 'misses': 2}
saved: dat

In [1]:
%%writefile tools/evidence.py
from typing import Dict, List
from urllib.parse import urlparse

MIN_INDEPENDENT_SOURCES = 2

# Fine to read, but not counted as independent corroboration.
LOW_QUALITY_DOMAINS = {
    "youtube.com", "youtu.be", "facebook.com", "tiktok.com", "pinterest.com",
    "instagram.com", "x.com", "twitter.com", "quora.com",
    # Added Day 26: user-generated posts are not independent corroboration.
    "linkedin.com", "reddit.com", "threads.net", "t.me", "snapchat.com",
}

_TWO_PART_SUFFIXES = {"co.uk", "com.au", "co.in", "com.cn", "co.jp", "com.br", "co.kr"}


def _registrable(host: str) -> str:
    host = host.lower().split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    parts = host.split(".")
    if len(parts) >= 3 and ".".join(parts[-2:]) in _TWO_PART_SUFFIXES:
        return ".".join(parts[-3:])
    if len(parts) >= 2:
        return ".".join(parts[-2:])
    return host


def source_domain(source: Dict) -> str:
    url = source.get("url", "") or ""
    host = urlparse(url).hostname or ""
    # Gemini grounding links redirect through a Google domain; the real site is in the title.
    if host.endswith("vertexaisearch.cloud.google.com"):
        title = (source.get("title") or "").strip().lower()
        if "." in title and " " not in title:
            return _registrable(title)
    return _registrable(host) if host else ""


def independent_domains(sources: List[Dict]) -> set:
    domains = {source_domain(s) for s in sources}
    domains.discard("")
    return domains - LOW_QUALITY_DOMAINS


def independent_source_count(sources: List[Dict]) -> int:
    return len(independent_domains(sources))

Overwriting tools/evidence.py


In [2]:
%%writefile tests/test_evidence_social.py
from tools.evidence import independent_domains, independent_source_count


def test_social_posts_do_not_count_as_independent_sources():
    sources = [{"url": "https://briefing.tfie.io/p/x", "title": "t"},
               {"url": "https://www.linkedin.com/posts/someone-activity-123", "title": "t"},
               {"url": "https://www.reddit.com/r/batteries/comments/abc", "title": "t"}]
    assert independent_domains(sources) == {"tfie.io"}
    assert independent_source_count(sources) == 1

Writing tests/test_evidence_social.py


In [3]:
%%writefile ui/progress.py
"""Progress tracking for the UI (Day 26). Pure Python (no Streamlit import) so it is unit-tested.
Turns LangGraph 'updates' chunks into readable log lines and a 5-stage pipeline status."""
import html
import time
from collections import Counter
from typing import Callable, Dict, List, Optional, Tuple

STAGE_CARDS = [
    ("web_search_single", "Search", "🔎"),
    ("data_extraction", "Extract", "🧾"),
    ("fact_checking", "Verify", "✅"),
    ("summarization", "Summarize", "📝"),
    ("report_generator", "Report", "📄"),
]
STAGES = [node for node, _, _ in STAGE_CARDS]
STAGE_LABELS = {
    "supervisor": "Supervisor",
    "web_search_single": "Web search",
    "data_extraction": "Data extraction",
    "fact_checking": "Fact-checking",
    "summarization": "Summarization",
    "report_generator": "Report generator",
}


def clip(text: str, n: int) -> str:
    """Shorten at a word boundary with an ellipsis."""
    text = " ".join((text or "").split())
    if len(text) <= n:
        return text
    cut = text[:n].rsplit(" ", 1)[0] or text[:n]
    return cut.rstrip(",;:- ") + "…"


def stage_progress(node: str) -> Optional[float]:
    """0..1 progress once `node` has reported (None for nodes outside the pipeline)."""
    if node not in STAGES:
        return None
    return (STAGES.index(node) + 1) / len(STAGES)


def describe_update(node: str, update) -> str:
    """One short line describing what a node just did."""
    update = update or {}
    errors = update.get("errors") or []
    suffix = f"  ⚠ {len(errors)} error(s)" if errors else ""

    if node == "supervisor":
        plan = update.get("plan")
        if plan:
            return f"Planned {len(plan)} sub-questions" + suffix
        nxt = update.get("next_agent")
        return (f"Next step: {STAGE_LABELS.get(nxt, nxt)}" if nxt else "Supervisor decided the next step") + suffix

    if node == "web_search_single":
        lines = []
        for r in update.get("search_results") or []:
            q = clip(r.get("sub_question") or "", 60)
            if r.get("status") == "failed" or not r.get("sources"):
                lines.append(f"Search found nothing for “{q}”")
            else:
                lines.append(f"Searched “{q}” via {r.get('provider', '?')} ({len(r['sources'])} sources)")
        return "; ".join(lines) + suffix if lines else "Search finished" + suffix

    if node == "data_extraction":
        return f"Extracted {len(update.get('extracted_facts') or [])} claims" + suffix

    if node == "fact_checking":
        results = update.get("fact_check_results") or []
        c = Counter(r.get("status", "UNVERIFIED") for r in results)
        return (f"Checked {len(results)} claims: {c.get('CONFIRMED', 0)} confirmed, "
                f"{c.get('DISPUTED', 0)} disputed, {c.get('UNVERIFIED', 0)} unverified") + suffix

    if node == "summarization":
        return f"Summarised {len(update.get('summaries') or [])} sub-topic(s)" + suffix

    if node == "report_generator":
        report = update.get("final_report") or ""
        return f"Report written ({len(report)} characters)" + suffix

    return f"{node} finished" + suffix


def iter_updates(chunk) -> List[Tuple[str, Dict]]:
    """Normalises one 'updates' chunk -> [(node, update)], tolerating None / list updates."""
    out = []
    if not isinstance(chunk, dict):
        return out
    for node, update in chunk.items():
        if isinstance(update, list):
            out.extend((node, u) for u in update)
        else:
            out.append((node, update))
    return out


class PipelineTracker:
    """Tracks which of the 5 pipeline stages is pending / running / done, with timings."""

    def __init__(self, clock: Callable[[], float] = time.time):
        self._clock = clock
        self._t0: Optional[float] = None
        self.stages = {n: {"status": "pending", "detail": "", "seconds": None, "started": None}
                       for n in STAGES}
        self.expected_searches: Optional[int] = None
        self._searches_done = 0
        self.events: List[Dict] = []

    def start(self) -> None:
        self._t0 = self._clock()
        self._begin(STAGES[0])

    def _begin(self, node: str) -> None:
        s = self.stages[node]
        if s["status"] == "pending":
            s["status"], s["started"] = "running", self._clock()

    def _finish(self, node: str, detail: str) -> None:
        s = self.stages[node]
        if s["status"] == "done":
            return
        if s["started"] is None:
            s["started"] = self._clock()
        s["status"], s["detail"] = "done", detail
        s["seconds"] = self._clock() - s["started"]
        idx = STAGES.index(node)
        if idx + 1 < len(STAGES):
            self._begin(STAGES[idx + 1])

    def on_update(self, node: str, update) -> str:
        update = update or {}
        text = describe_update(node, update)
        elapsed = (self._clock() - self._t0) if self._t0 is not None else 0.0
        self.events.append({"t": round(elapsed, 1), "node": node,
                            "label": STAGE_LABELS.get(node, node), "text": text})
        if node == "supervisor" and update.get("plan"):
            self.expected_searches = len(update["plan"])
        if node == "web_search_single":
            self._searches_done += 1
            target = self.expected_searches or 1
            if self._searches_done >= target:
                self._finish(node, f"{self._searches_done} queries")
            else:
                self.stages[node]["detail"] = f"{self._searches_done}/{target} queries"
        elif node in self.stages:
            self._finish(node, text)
        return text

    def fraction_done(self) -> float:
        return sum(1 for s in self.stages.values() if s["status"] == "done") / len(STAGES)

    def snapshot(self) -> List[Dict]:
        return [{"node": n, "label": label, "icon": icon, **self.stages[n]}
                for n, label, icon in STAGE_CARDS]


_PIPELINE_CSS = """<style>
.pl-wrap{display:flex;gap:10px;flex-wrap:wrap;margin:6px 0 14px 0}
.pl-card{flex:1;min-width:140px;border:1px solid rgba(128,128,128,.35);border-radius:12px;padding:12px 14px}
.pl-title{font-weight:600;font-size:1rem}
.pl-status{font-size:.82rem;margin-top:2px}
.pl-detail{font-size:.75rem;opacity:.7;margin-top:4px;line-height:1.25}
.pl-pending{opacity:.5}
.pl-running{border-color:#3b82f6;box-shadow:0 0 0 2px rgba(59,130,246,.25);animation:pl-pulse 1.4s infinite}
.pl-done{border-color:#22c55e;background:rgba(34,197,94,.08)}
@keyframes pl-pulse{0%{box-shadow:0 0 0 0 rgba(59,130,246,.45)}70%{box-shadow:0 0 0 8px rgba(59,130,246,0)}100%{box-shadow:0 0 0 0 rgba(59,130,246,0)}}
</style>"""


def render_pipeline_html(stages: List[Dict]) -> str:
    """HTML for the stage cards (use with st.markdown(..., unsafe_allow_html=True))."""
    cards = []
    for s in stages:
        status = s["status"]
        if status == "done":
            line = f"Done · {s['seconds']:.1f}s" if s.get("seconds") is not None else "Done"
        elif status == "running":
            line = "Running…"
        else:
            line = "Waiting"
        detail = html.escape(clip(s.get("detail") or "", 70))
        cards.append(
            f'<div class="pl-card pl-{status}"><div class="pl-title">{s["icon"]} {html.escape(s["label"])}</div>'
            f'<div class="pl-status">{line}</div><div class="pl-detail">{detail}</div></div>')
    return _PIPELINE_CSS + '<div class="pl-wrap">' + "".join(cards) + "</div>"

Overwriting ui/progress.py


In [4]:
%%writefile ui/views.py
"""Pure data-shaping helpers for the UI (Day 26) - no Streamlit import, fully unit-tested."""
import html as _html
from typing import Dict, List, Optional

STATUS_ORDER = ["CONFIRMED", "DISPUTED", "UNVERIFIED"]
STATUS_BADGE = {"CONFIRMED": "🟢", "DISPUTED": "🔴", "UNVERIFIED": "🟡"}

GRAPH_DOT = """
digraph G {
  rankdir=LR; bgcolor="transparent";
  node [shape=box, style="rounded,filled", fillcolor="#eef2ff", color="#6366f1", fontname="Helvetica"];
  edge [color="#6b7280", fontname="Helvetica", fontsize=10];
  start [label="Question", shape=oval, fillcolor="#f3f4f6", color="#9ca3af"];
  supervisor [label="Supervisor\\n(plan + route)", fillcolor="#e0e7ff"];
  search [label="Web search\\n(parallel fan-out)"];
  extract [label="Data extraction"];
  verify [label="Fact-checking\\n(merge + verify + gate)"];
  summarize [label="Summarization"];
  report [label="Report generator"];
  end [label="Report + evidence", shape=oval, fillcolor="#dcfce7", color="#22c55e"];
  start -> supervisor;
  supervisor -> search [label="Send ×N"];
  supervisor -> extract; supervisor -> verify; supervisor -> summarize; supervisor -> report;
  search -> supervisor [style=dashed]; extract -> supervisor [style=dashed];
  verify -> supervisor [style=dashed]; summarize -> supervisor [style=dashed];
  report -> end;
}
"""


def short_topic(question: str, n: int = 52) -> str:
    text = " ".join((question or "").split())
    if len(text) <= n:
        return text
    return (text[:n].rsplit(" ", 1)[0] or text[:n]).rstrip(",;:- ") + "…"


def topic_options(results: List[Dict]) -> List[str]:
    """Distinct sub-questions in order of first appearance."""
    seen, out = set(), []
    for r in results:
        q = r.get("sub_question")
        if q and q not in seen:
            seen.add(q)
            out.append(q)
    return out


def claim_rows(results: List[Dict]) -> List[Dict]:
    rows = []
    for r in results:
        rows.append({
            "claim": r.get("claim", ""),
            "status": r.get("status", "UNVERIFIED"),
            "topic": r.get("sub_question", ""),
            "domains": r.get("independent_sources", 0),
            "sources": list(r.get("sources") or []),
            "verified": bool(r.get("verified_by_search")),
            "merged": r.get("merged_count", 1),
            "reasoning": r.get("reasoning", ""),
            "downgraded": bool(r.get("downgraded")),
        })
    return rows


def filter_rows(rows: List[Dict], statuses: Optional[List[str]] = None, topic: Optional[str] = None,
                verified_only: bool = False, text: str = "") -> List[Dict]:
    text = (text or "").strip().lower()
    out = []
    for r in rows:
        if statuses and r["status"] not in statuses:
            continue
        if topic and r["topic"] != topic:
            continue
        if verified_only and not r["verified"]:
            continue
        if text and text not in r["claim"].lower():
            continue
        out.append(r)
    return out


def status_counts(results: List[Dict]) -> Dict[str, int]:
    counts = {s: 0 for s in STATUS_ORDER}
    for r in results:
        counts[r.get("status", "UNVERIFIED")] = counts.get(r.get("status", "UNVERIFIED"), 0) + 1
    return counts


def confirmation_rate(results: List[Dict]) -> float:
    return (status_counts(results)["CONFIRMED"] / len(results)) if results else 0.0


def status_by_topic(results: List[Dict]) -> List[Dict]:
    """One row per sub-topic: {"topic": "Q1: …", "CONFIRMED": n, "DISPUTED": n, "UNVERIFIED": n}."""
    rows = []
    for i, q in enumerate(topic_options(results), start=1):
        row = {"topic": f"Q{i}: {short_topic(q, 36)}", **{s: 0 for s in STATUS_ORDER}}
        for r in results:
            if r.get("sub_question") == q:
                st_ = r.get("status", "UNVERIFIED")
                row[st_] = row.get(st_, 0) + 1
        rows.append(row)
    return rows


def node_rows(metrics_summary: Dict) -> List[Dict]:
    rows = []
    for node, n in (metrics_summary.get("nodes") or {}).items():
        seconds = n.get("node_seconds") or n.get("llm_seconds") or 0.0
        if seconds or n.get("calls"):
            rows.append({"node": node, "seconds": round(seconds, 1), "output_tokens": n.get("output", 0),
                         "llm_calls": n.get("calls", 0), "rate_limited": n.get("rate_limited", 0)})
    return rows


def source_html(source: Dict) -> str:
    """HTML for one source line. Titles/URLs/snippets come from the open web, so everything is
    escaped and only http(s) links are allowed."""
    raw_url = source.get("url", "") or ""
    url = _html.escape(raw_url, quote=True) if raw_url.startswith(("http://", "https://")) else "#"
    title = _html.escape(source.get("title") or raw_url or "source")
    tag = " <em>(verification)</em>" if source.get("via") == "verification" else ""
    out = f'<div class="src">• <a href="{url}" target="_blank" rel="noopener noreferrer">{title}</a>{tag}</div>'
    if source.get("evidence"):
        out += f'<div class="quote">“{_html.escape(source["evidence"])}”</div>'
    return out

Writing ui/views.py


In [5]:
%%writefile ui/history.py
"""Saves each UI run as a JSON snapshot so past runs can be reopened (Day 26)."""
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

HISTORY_DIR = Path("data/history")
SCHEMA_VERSION = 1


def save_snapshot(snapshot: Dict, directory: Optional[Path] = None) -> Path:
    directory = Path(directory or HISTORY_DIR)
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / f"{datetime.now():%Y%m%d_%H%M%S_%f}.json"
    path.write_text(json.dumps({"schema_version": SCHEMA_VERSION, **snapshot}, indent=2, default=str),
                    encoding="utf-8")
    return path


def load_snapshot(path) -> Dict:
    return json.loads(Path(path).read_text(encoding="utf-8"))


def list_snapshots(directory: Optional[Path] = None) -> List[Dict]:
    """Newest first. Unreadable files are skipped rather than breaking the app."""
    directory = Path(directory or HISTORY_DIR)
    if not directory.exists():
        return []
    out = []
    for p in sorted(directory.glob("*.json"), reverse=True):
        try:
            d = load_snapshot(p)
        except (OSError, ValueError):
            continue
        results = d.get("fact_check_results") or []
        out.append({"path": p, "created_at": d.get("created_at", ""), "query": d.get("query", ""),
                    "claims": len(results),
                    "confirmed": sum(1 for r in results if r.get("status") == "CONFIRMED")})
    return out

Writing ui/history.py


In [6]:
%%writefile tests/test_progress.py
from ui.progress import (PipelineTracker, clip, describe_update, iter_updates,
                         render_pipeline_html, stage_progress)


def test_supervisor_plan_and_routing():
    assert describe_update("supervisor", {"plan": ["a", "b", "c"]}) == "Planned 3 sub-questions"
    assert describe_update("supervisor", {"next_agent": "fact_checking"}) == "Next step: Fact-checking"
    assert describe_update("supervisor", None) == "Supervisor decided the next step"


def test_search_line_reports_provider_and_failures():
    ok = {"search_results": [{"sub_question": "How does X work?", "provider": "tavily", "sources": [1, 2, 3, 4]}]}
    bad = {"search_results": [{"sub_question": "Q", "provider": "none", "sources": [], "status": "failed"}]}
    assert "via tavily (4 sources)" in describe_update("web_search_single", ok)
    assert "found nothing" in describe_update("web_search_single", bad)


def test_fact_check_counts_statuses():
    u = {"fact_check_results": [{"status": "CONFIRMED"}, {"status": "UNVERIFIED"}, {"status": "UNVERIFIED"}]}
    assert describe_update("fact_checking", u) == "Checked 3 claims: 1 confirmed, 0 disputed, 2 unverified"


def test_errors_are_flagged_and_unknown_nodes_fall_back():
    assert "⚠ 1 error(s)" in describe_update("data_extraction", {"extracted_facts": [], "errors": [{"x": 1}]})
    assert describe_update("mystery", {}) == "mystery finished"


def test_progress_fraction_only_for_pipeline_nodes():
    assert stage_progress("supervisor") is None
    assert stage_progress("web_search_single") == 0.2 and stage_progress("report_generator") == 1.0


def test_iter_updates_tolerates_odd_shapes():
    assert iter_updates({"a": {"x": 1}, "b": None}) == [("a", {"x": 1}), ("b", None)]
    assert iter_updates({"a": [{"x": 1}, {"x": 2}]}) == [("a", {"x": 1}), ("a", {"x": 2})]
    assert iter_updates("not a dict") == []


def test_clip_breaks_on_word_boundary():
    assert clip("What are the primary technical processes and recovery rates for lithium", 40) == \
        "What are the primary technical…"
    assert clip("short", 40) == "short"


class FakeClock:
    def __init__(self):
        self.t = 100.0

    def __call__(self):
        return self.t


def test_tracker_walks_through_stages_with_timings():
    clock = FakeClock()
    t = PipelineTracker(clock=clock)
    t.start()
    assert t.stages["web_search_single"]["status"] == "running"

    t.on_update("supervisor", {"plan": ["a", "b"]})
    clock.t += 2
    t.on_update("web_search_single", {"search_results": [{"sub_question": "a", "sources": [1]}]})
    assert t.stages["web_search_single"]["status"] == "running"          # 1 of 2 searches
    assert t.stages["web_search_single"]["detail"] == "1/2 queries"
    clock.t += 1
    t.on_update("web_search_single", {"search_results": [{"sub_question": "b", "sources": [1]}]})
    assert t.stages["web_search_single"]["status"] == "done"
    assert t.stages["web_search_single"]["seconds"] == 3.0
    assert t.stages["data_extraction"]["status"] == "running"            # next stage starts automatically

    clock.t += 10
    t.on_update("data_extraction", {"extracted_facts": [1, 2, 3]})
    assert t.stages["data_extraction"]["seconds"] == 10.0
    assert t.fraction_done() == 0.4
    assert [e["node"] for e in t.events][:2] == ["supervisor", "web_search_single"]


def test_render_html_shows_each_state_and_escapes_text():
    clock = FakeClock()
    t = PipelineTracker(clock=clock)
    t.start()
    t.stages["web_search_single"]["detail"] = "<script>alert(1)</script>"
    html_out = render_pipeline_html(t.snapshot())
    assert "pl-running" in html_out and "pl-pending" in html_out
    assert "<script>alert" not in html_out and "&lt;script&gt;" in html_out

Overwriting tests/test_progress.py


In [7]:
%%writefile tests/test_views.py
from ui.views import (claim_rows, confirmation_rate, filter_rows, node_rows, short_topic,
                      status_by_topic, status_counts, topic_options)

RESULTS = [
    {"claim": "Hydrometallurgy recovers 95% of lithium", "status": "CONFIRMED", "sub_question": "How do processes work in recycling?",
     "independent_sources": 2, "sources": [{"url": "https://a.org"}, {"url": "https://b.com", "via": "verification"}],
     "verified_by_search": 1},
    {"claim": "Redwood is in Nevada", "status": "UNVERIFIED", "sub_question": "How do processes work in recycling?",
     "independent_sources": 1, "sources": [{"url": "https://a.org"}]},
    {"claim": "China hosts 85% of capacity", "status": "UNVERIFIED", "sub_question": "What do regulations say?",
     "independent_sources": 1, "sources": []},
    {"claim": "Rules conflict", "status": "DISPUTED", "sub_question": "What do regulations say?", "merged_count": 2},
]


def test_counts_and_rate():
    assert status_counts(RESULTS) == {"CONFIRMED": 1, "DISPUTED": 1, "UNVERIFIED": 2}
    assert confirmation_rate(RESULTS) == 0.25 and confirmation_rate([]) == 0.0


def test_status_by_topic_has_one_row_per_subquestion():
    rows = status_by_topic(RESULTS)
    assert [r["topic"][:3] for r in rows] == ["Q1:", "Q2:"]
    assert rows[0]["CONFIRMED"] == 1 and rows[0]["UNVERIFIED"] == 1
    assert rows[1]["DISPUTED"] == 1 and rows[1]["UNVERIFIED"] == 1


def test_filters_combine():
    rows = claim_rows(RESULTS)
    assert len(filter_rows(rows)) == 4
    assert [r["claim"] for r in filter_rows(rows, statuses=["UNVERIFIED"])] == ["Redwood is in Nevada", "China hosts 85% of capacity"]
    assert len(filter_rows(rows, verified_only=True)) == 1
    assert len(filter_rows(rows, topic=topic_options(RESULTS)[1], statuses=["DISPUTED"])) == 1
    assert [r["claim"] for r in filter_rows(rows, text="NEVADA")] == ["Redwood is in Nevada"]


def test_claim_rows_are_defensive_about_missing_fields():
    row = claim_rows([{"claim": "x"}])[0]
    assert row["status"] == "UNVERIFIED" and row["sources"] == [] and row["merged"] == 1 and not row["verified"]


def test_short_topic_and_node_rows():
    assert short_topic("a " * 40, 20).endswith("…") and short_topic("short") == "short"
    rows = node_rows({"nodes": {"data_extraction": {"node_seconds": 58.14, "output": 1937, "calls": 4, "rate_limited": 0},
                                "idle": {}}})
    assert rows == [{"node": "data_extraction", "seconds": 58.1, "output_tokens": 1937, "llm_calls": 4, "rate_limited": 0}]


def test_source_html_escapes_web_content_and_blocks_bad_links():
    from ui.views import source_html
    out = source_html({"url": "javascript:alert(1)", "title": "<img src=x onerror=alert(1)>", "evidence": "a <b>quote</b>"})
    assert 'href="#"' in out and "<img" not in out and "&lt;img" in out and "<b>" not in out
    ok = source_html({"url": "https://a.org/x?q=1&r=2", "title": "A", "via": "verification", "evidence": "the quote"})
    assert 'href="https://a.org/x?q=1&amp;r=2"' in ok and "(verification)" in ok and "the quote" in ok

Writing tests/test_views.py


In [8]:
%%writefile tests/test_history.py
from ui.history import list_snapshots, load_snapshot, save_snapshot


def test_save_list_and_load_roundtrip(tmp_path):
    snap = {"query": "q1", "created_at": "2026-11-03T18:00:00",
            "fact_check_results": [{"status": "CONFIRMED"}, {"status": "UNVERIFIED"}], "final_report": "# R"}
    path = save_snapshot(snap, tmp_path)
    assert load_snapshot(path)["final_report"] == "# R"
    rows = list_snapshots(tmp_path)
    assert len(rows) == 1 and rows[0]["claims"] == 2 and rows[0]["confirmed"] == 1 and rows[0]["query"] == "q1"


def test_newest_first_and_corrupt_files_skipped(tmp_path):
    (tmp_path / "20260101_000000_000000.json").write_text("{not json")
    save_snapshot({"query": "older"}, tmp_path)
    save_snapshot({"query": "newer"}, tmp_path)
    assert [r["query"] for r in list_snapshots(tmp_path)][0] == "newer"
    assert len(list_snapshots(tmp_path)) == 2


def test_missing_directory_is_empty(tmp_path):
    assert list_snapshots(tmp_path / "nope") == []

Writing tests/test_history.py


In [9]:
import os
os.makedirs(".streamlit", exist_ok=True)
open(".streamlit/config.toml", "w", encoding="utf-8").write('''[theme]
primaryColor = "#4f46e5"
font = "sans serif"

[browser]
gatherUsageStats = false
''')
with open(".gitignore", "a") as f:
    f.write("\ndata/history/\n")
print("done")

done


In [ ]:
%%writefile app.py
"""Multi-Agent Research & Intelligence System - Streamlit front end.

Run from a terminal in the project root:   streamlit run app.py
Live agent pipeline, evidence explorer (every claim with its sources and the sentence that backs it),
performance metrics, run history and export. All logic lives in tested modules (ui/, tools/)."""
import json
import os
import time
from datetime import datetime

import pandas as pd
import streamlit as st
from dotenv import load_dotenv

load_dotenv()

from tools import cache  # noqa: E402
from tools.metrics import RunMetrics, evidence_stats  # noqa: E402
from ui import history, views  # noqa: E402
from ui.progress import (PipelineTracker, iter_updates, render_pipeline_html)  # noqa: E402

st.set_page_config(page_title="Multi-Agent Research System", page_icon="🔬", layout="wide")

PRESETS = [
    "Compare battery recycling approaches of Tesla, Redwood Materials and CATL",
    "What are the safety risks of solid-state EV batteries?",
    "What is CATL's sodium-ion battery strategy and how does it compare to lithium-ion?",
]
REQUIRED_KEYS = ("GROQ_API_KEY", "GEMINI_API_KEY", "TAVILY_API_KEY")

st.markdown("""<style>
.hero h1{margin-bottom:0}
.hero p{opacity:.75;margin-top:.25rem}
.badge{display:inline-block;padding:2px 10px;margin:0 6px 6px 0;border-radius:999px;font-size:.75rem;
       border:1px solid rgba(128,128,128,.4)}
.src{font-size:.85rem;margin:2px 0}
.quote{font-size:.8rem;opacity:.75;margin:0 0 6px 18px}
</style>""", unsafe_allow_html=True)

st.session_state.setdefault("query", PRESETS[0])


def set_query(q: str) -> None:
    st.session_state["query"] = q


@st.cache_resource
def get_graph():
    from graph.build_graph import build_graph
    return build_graph()


# ------------------------------------------------------------------ sidebar
with st.sidebar:
    st.markdown("### 🔬 Research Intelligence")
    st.caption("Supervisor-orchestrated multi-agent pipeline built with LangGraph.")

    st.markdown("**Example questions**")
    for i, q in enumerate(PRESETS):
        st.button(q, key=f"preset{i}", on_click=set_query, args=(q,))

    st.markdown("**Options**")
    verify = st.checkbox("Verify single-source claims by search", value=True,
                         help="Searches for a second independent source for up to 6 claims per run, "
                              "spread across sub-topics.")
    use_cache = st.checkbox("Cache search results (24h)", value=True,
                            help="Repeated queries skip the search APIs.")

    st.markdown("**Past runs**")
    snaps = history.list_snapshots()
    if snaps:
        labels = [f"{s['created_at'][5:16].replace('T', ' ')} · {s['confirmed']}/{s['claims']} confirmed · "
                  f"{views.short_topic(s['query'], 34)}" for s in snaps]
        choice = st.selectbox("Load a saved run", range(len(snaps)), format_func=lambda i: labels[i])
        if st.button("Open selected run"):
            st.session_state["snapshot"] = history.load_snapshot(snaps[choice]["path"])
    else:
        st.caption("Runs are saved here automatically.")

# ------------------------------------------------------------------ hero + input
st.markdown("""<div class="hero"><h1>Multi-Agent Research & Intelligence System</h1>
<p>Ask a research question. Specialised agents search, extract, fact-check, summarise and write a report,
and every claim is traced to its sources.</p>
<span class="badge">LangGraph</span><span class="badge">Supervisor + parallel fan-out</span>
<span class="badge">Groq · Qwen</span><span class="badge">Gemini → Tavily → DuckDuckGo</span>
<span class="badge">2-independent-source gate</span><span class="badge">Evidence verification</span></div>""",
            unsafe_allow_html=True)

st.text_area("Research question", key="query", height=80)
c_run, c_note = st.columns([1, 5])
run_clicked = c_run.button("▶ Run research", type="primary", disabled=not st.session_state["query"].strip())
c_note.caption("A full run takes about 2–4 minutes on free-tier APIs. Cached searches are instant.")


# ------------------------------------------------------------------ run
def run_pipeline(query: str):
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    if missing:
        st.error(f"Missing API keys in .env: {', '.join(missing)}")
        return None
    os.environ["VERIFY_BY_SEARCH"] = "true" if verify else "false"
    os.environ["SEARCH_CACHE"] = "true" if use_cache else "false"
    try:
        graph = get_graph()
    except Exception as e:
        st.error(f"Could not start the graph: {type(e).__name__}: {e}")
        return None

    metrics, tracker = RunMetrics(), PipelineTracker()
    initial_state = {
        "query": query, "plan": [], "next_agent": "", "iteration_count": 0,
        "needs_more_research": False, "search_results": [], "extracted_facts": [],
        "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
        "model_usage_log": [],
    }
    holder = st.empty()
    bar = st.progress(0.0)
    tracker.start()
    holder.markdown(render_pipeline_html(tracker.snapshot()), unsafe_allow_html=True)
    cache_before, started, final = cache.stats(), time.time(), None
    try:
        with st.expander("Agent activity", expanded=True):
            for mode, chunk in graph.stream(initial_state, config={"callbacks": [metrics]},
                                            stream_mode=["updates", "values"]):
                if mode == "values":
                    final = chunk
                    continue
                for node, update in iter_updates(chunk):
                    text = tracker.on_update(node, update)
                    st.write(f"`{tracker.events[-1]['t']:>6.1f}s`  **{tracker.events[-1]['label']}** — {text}")
                    holder.markdown(render_pipeline_html(tracker.snapshot()), unsafe_allow_html=True)
                    bar.progress(tracker.fraction_done())
    except Exception as e:
        hint = " The free-tier model is rate limited; wait a minute and retry." if "429" in str(e) else ""
        st.error(f"Run failed: {type(e).__name__}: {e}.{hint}")
        return None
    bar.progress(1.0)
    if not final:
        st.error("The graph finished without returning a final state.")
        return None

    cache_after = cache.stats()
    snapshot = {
        "query": query, "created_at": datetime.now().isoformat(timespec="seconds"),
        "final_report": final.get("final_report") or "", "fact_check_results": final.get("fact_check_results", []),
        "summaries": final.get("summaries", []), "metrics": metrics.summary(), "evidence": evidence_stats(final),
        "errors": [list(e) for e in metrics.error_log], "stages": tracker.snapshot(), "events": tracker.events,
        "options": {"verify": verify, "cache": use_cache},
        "cache": {k: cache_after[k] - cache_before[k] for k in cache_after},
        "wall_seconds": round(time.time() - started, 1),
    }
    history.save_snapshot(snapshot)
    return snapshot


if run_clicked:
    snap = run_pipeline(st.session_state["query"].strip())
    if snap:
        st.session_state["snapshot"] = snap
        st.success(f"Finished in {snap['wall_seconds']:.0f}s")


# ------------------------------------------------------------------ results
def render_results(snap: dict) -> None:
    results = snap.get("fact_check_results", [])
    counts = views.status_counts(results)
    m = snap.get("metrics", {})
    ev = snap.get("evidence", {})

    st.divider()
    st.markdown(f"#### Results for: *{snap.get('query', '')}*")
    k = st.columns(5)
    k[0].metric("Claims checked", len(results))
    k[1].metric("Confirmed", counts["CONFIRMED"], f"{views.confirmation_rate(results):.0%} of claims", delta_color="off")
    k[2].metric("Upgraded by verification", ev.get("claims_verified_by_search", 0))
    k[3].metric("Run time", f"{m.get('total_seconds', 0):.0f}s")
    k[4].metric("Rate-limit errors", m.get("rate_limit_errors", 0))

    t_report, t_evidence, t_perf, t_about = st.tabs(["📄 Report", "🔍 Evidence explorer", "⚙️ Pipeline & performance", "ℹ️ How it works"])

    with t_report:
        report = snap.get("final_report") or "_No report was produced._"
        st.markdown(report)
        d1, d2 = st.columns(2)
        d1.download_button("⬇ Report (.md)", report, file_name="report.md", mime="text/markdown")
        d2.download_button("⬇ Full run data (.json)", json.dumps(snap, indent=2, default=str),
                           file_name="run.json", mime="application/json")
        for s in snap.get("summaries") or []:
            if s.get("summary"):
                with st.expander(views.short_topic(s.get("sub_question", "Sub-topic"), 90)):
                    st.write(s["summary"])

    with t_evidence:
        rows = views.claim_rows(results)
        by_topic = views.status_by_topic(results)
        if by_topic:
            st.caption("Claims per sub-topic by verdict")
            st.bar_chart(pd.DataFrame(by_topic).set_index("topic"))
        f1, f2, f3, f4 = st.columns([2, 3, 1.4, 2])
        statuses = f1.multiselect("Verdict", views.STATUS_ORDER, default=views.STATUS_ORDER)
        topics = views.topic_options(results)
        topic = f2.selectbox("Sub-topic", [None] + topics, format_func=lambda q: "All" if q is None else views.short_topic(q, 60))
        verified_only = f3.checkbox("Verified by search")
        text = f4.text_input("Search claims")
        shown = views.filter_rows(rows, statuses=statuses, topic=topic, verified_only=verified_only, text=text)
        st.caption(f"Showing {len(shown)} of {len(rows)} claims")
        for r in shown:
            badge = views.STATUS_BADGE.get(r["status"], "⚪")
            with st.expander(f"{badge} {r['claim']}"):
                meta = [f"**{r['status']}**", f"{r['domains']} independent domain(s)"]
                if r["verified"]:
                    meta.append("extra source found by verification search")
                if r["merged"] > 1:
                    meta.append(f"merged from {r['merged']} duplicate claims")
                if r["downgraded"]:
                    meta.append("downgraded by the evidence gate")
                st.markdown(" · ".join(meta))
                if r["reasoning"]:
                    st.caption(r["reasoning"])
                for s in r["sources"]:
                    st.markdown(views.source_html(s), unsafe_allow_html=True)

    with t_perf:
        st.markdown("**Pipeline run**")
        st.markdown(render_pipeline_html(snap.get("stages") or []), unsafe_allow_html=True)
        node_df = pd.DataFrame(views.node_rows(m))
        if not node_df.empty:
            c1, c2 = st.columns(2)
            c1.caption("Seconds per node")
            c1.bar_chart(node_df.set_index("node")["seconds"])
            c2.caption("Output tokens per node")
            c2.bar_chart(node_df.set_index("node")["output_tokens"])
            st.dataframe(node_df, hide_index=True)
        p = st.columns(4)
        p[0].metric("Output tokens", m.get("total_output_tokens", 0))
        p[1].metric("Peak tokens / 60s", m.get("peak_output_tokens_60s", 0), help="Groq free tier limit: 1000")
        cache_info = snap.get("cache") or {}
        p[2].metric("Cache hits", cache_info.get("hits", 0))
        p[3].metric("Cache misses", cache_info.get("misses", 0))
        for node, kind, msg in snap.get("errors") or []:
            st.warning(f"LLM error in {node} ({kind}): {msg}")
        with st.expander("Agent activity log"):
            for e in snap.get("events") or []:
                st.write(f"`{e['t']:>6.1f}s`  **{e['label']}** — {e['text']}")

    with t_about:
        st.markdown("""**Pipeline.** A Supervisor decomposes the question into sub-questions and routes work
deterministically by data dependency. Searches run in parallel (bounded by a semaphore), then extraction,
fact-checking, summarisation and report generation run in order. Every node logs on every path, so a failure
degrades a result instead of silently dropping it.""")
        st.graphviz_chart(views.GRAPH_DOT)
        st.markdown("""**What "confirmed" means.** A claim is CONFIRMED only if sources from at least two
*independent domains* back it (social platforms excluded). Duplicate claims across sub-questions are merged and
their sources pooled. Claims with fewer sources get a targeted verification search for a second domain, and a new
source is accepted only if one of its sentences states the same figures. A code-enforced gate downgrades any
CONFIRMED verdict that lacks two independent domains, whatever the model said.

**Known limits.** Independent *domains* are not always independent *origins*: two sites can repeat the same
upstream statistic. Free-tier rate limits make runs slow and occasionally degrade a step to UNVERIFIED. Extraction
is not fully deterministic, so claim counts vary a little between runs.""")


snapshot = st.session_state.get("snapshot")
if snapshot:
    render_results(snapshot)
else:
    st.info("Pick an example or type a question, then press **Run research**.")